# Weenies and Buns: indexed resource limits

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gromicho/teaching/blob/main/courses/abw/notebooks/optimization/weenies-and-buns.ipynb) [![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/courses/abw/notebooks/optimization/weenies-and-buns.ipynb)

**Exercise** · UvA Business Analytics teaching collection

Work through the questions before running each cell. Explain the result, check its assumptions, and change one input to test your understanding.


## Setup
Use the installed scientific libraries and install only missing dependencies. The shared helper keeps this routine code in one place. Solver-specific lessons introduce additional solvers at the point where they are used.


In [ ]:
# Load the shared teaching utilities from this checkout or a verified download.
from pathlib import Path
import hashlib
import sys
from urllib.request import urlopen

support_path = next((folder / 'support' for folder in [Path.cwd(), *Path.cwd().parents]
                     if (folder / 'support' / 'teaching_utils.py').is_file()), None)
if support_path is None:
    support_path = Path.cwd() / '.teaching-support'
    support_path.mkdir(exist_ok=True)
    helper = support_path / 'teaching_utils.py'
    expected = 'fbfa41e12709a01548e213abba976dad1e21d0abcfd798a2dfd669706cc152bb'
    if not helper.exists() or hashlib.sha256(helper.read_bytes()).hexdigest() != expected:
        url = 'https://raw.githubusercontent.com/gromicho/teaching/f3ad11b77cae7dd05315d314c7e72ee8516aaa3d/support/teaching_utils.py'
        content = urlopen(url, timeout=45).read()
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError('Teaching helper version changed; reopen the current course notebook.')
        helper.write_bytes(content)
sys.path.insert(0, str(support_path))
from teaching_utils import ensure_packages

required_packages = {'highspy': 'highspy', 'pyomo': 'pyomo'}
ensure_packages(required_packages)


In [ ]:
import pyomo.environ as pyo
from pyomo.opt import assert_optimal_termination
SOLVER = "appsi_highs"
assert pyo.SolverFactory(SOLVER).available(exception_flag=False), 'HiGHS is unavailable; rerun the setup cell and inspect its installation messages.'


## Data and decisions
Use the associated exercise statement in the course booklet. The coefficient arrays below preserve the indexed notebook data. Check that the product order is consistent everywhere; do not infer labels from an array position without consulting its definition.


In [ ]:
c = [0.88,0.33]
A = [[0,0.1],[0.25,0],[3,2]]
b = [200,800,12000]
products = range(len(c))
resources = range(len(b))


## Your formulation
Name the decisions and their units. Write the objective, every constraint, and the variable domains on paper before translating them into Pyomo. Inspect the resulting model before solving it. Finally, explain a feasibility check independently of the solver.

> **Optional UvA AI Chat prompt:** Ask me to explain my variables and units first. Then review one constraint at a time. If it is wrong, ask a question that helps me locate the mistake. Do not write the finished formulation for me.


In [ ]:
# TODO: build your indexed model and add your checks here.


## Check your plan independently

After formulating and solving your model, pass a dictionary of numerical decisions
to `check_plan`, with exactly the keys in `products`. If your indexed decision
variable is named `model.x`, the example below shows how to extract its values.
Adapt the model and variable names to your own formulation.

The checker recomputes each condition from the exercise data, independently of
your Pyomo constraints. Read the code: identify the units of each calculation and
explain its direction. The code stays visible because writing these checks is part
of learning to model. A tolerance of $10^{-6}$ allows small rounding errors in
the stated units; do not round your decisions before checking them.

**A feasible plan is not necessarily optimal.** First require optimal solver
termination, then compare the independently recomputed objective with your model's
objective. A solver can optimize an incorrectly formulated model, so both checks
matter. The checker also works on a hand-calculated plan without a solver.


In [ ]:
from math import isfinite


def check_plan(plan, tol=1e-6):
    """Report feasibility from the case data, without inspecting a Pyomo model."""
    if set(plan) != set(products):
        raise ValueError('Use exactly the keys in products; check spelling and indexing.')
    plan = {key: float(value) for key, value in plan.items()}
    if not all(isfinite(value) for value in plan.values()):
        raise ValueError('Every decision must have a finite numerical value.')
    if not isfinite(tol) or tol < 0:
        raise ValueError('The checking tolerance must be finite and nonnegative.')
    checks = {'Every decision is nonnegative': min(plan.values()) >= -tol}
    for resource in resources:
        used = sum(A[resource][j] * plan[j] for j in products)
        checks[f'Resource {resource}: use {used:g} <= {b[resource]:g}'] = used <= b[resource] + tol
    objective = sum(c[j] * plan[j] for j in products)
    for description, passed in checks.items():
        print(('OK: ' if passed else 'CHECK: ') + description)
    print(f'Independently computed objective: {objective:g}')
    feasible = all(checks.values())
    print('Feasibility checks passed; assess optimality separately.' if feasible
          else 'Some conditions fail. Review the indicated units or constraints.')
    return {'feasible': feasible, 'objective': objective, 'checks': checks}


In [ ]:
# Once your model is complete, adapt these names and uncomment:
# from teaching_utils import solve_checked
# results = solve_checked(model, SOLVER)
# plan = {key: pyo.value(model.x[key]) for key in products}
# feedback = check_plan(plan)
